In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# Deep Neural Networks

## Session 18a

## Simple RNN - Multiple Features
Weather Data 
- Multiple features

<img src='../../prasami_images/prasami_color_tutorials_small.png' width='400' alt="By Pramod Sharma : pramod.sharma@prasami.com" align="left"/>

In [ ]:
###-----------------
### Import Libraries
###-----------------

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
import tensorflow as tf

from utils.helper import fn_plot_tf_hist

In [ ]:
###----------------------
### Some basic parameters
###----------------------

inpDir = Path('../../input') # location where input data is stored
outDir = Path('../output') # location to store outputs

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE) # Set Random Seed for reproducible results
tf.random.set_seed(RANDOM_STATE) # setting for Tensorflow as well

EPOCHS = 50  # number of cycles to run
ALPHA = 0.001  # learning rate
TEST_SIZE = 0.2 # What fraction we want to keep for testing
BATCH_SIZE = 32

# Set parameters for decoration of plots
params = {'legend.fontsize': 'medium',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'large',
          'axes.titlesize':'large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }
CMAP = plt.cm.coolwarm

plt.rcParams.update(params) # update rcParams

## Helper Function

In [ ]:
### Settings so that Tensorflow can not Hog all the GPU memory
physical_devices = tf.config.list_physical_devices('GPU') 

if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

## Load Beijing PM2.5 Air Quality Dataset (UCI)
Source: UCI Archives

In [ ]:
dataFilename = 'PRSA_data_2010.1.1-2014.12.31.csv'
data_df = pd.read_csv(inpDir / dataFilename)
data_df.head()

In [ ]:
data_df.shape

In [ ]:
data_df.info()

In [ ]:
data_df.describe()

In [ ]:
data_df['datetime'] = pd.to_datetime(data_df[['year', 'month', 'day', 'hour']])
data_df = data_df.sort_values('datetime').reset_index(drop=True)
data_df.head()

In [ ]:
data_df.isnull().sum()

### Null value strategy.
pm2.5 has nulll values. Forward fill and backfill should work

In [ ]:
data_df = data_df.ffill()
data_df = data_df.bfill()
data_df.isnull().sum()

In [ ]:
data_df.columns

In [ ]:
rel_columns = ['pm2.5',   # Target variable
               'DEWP',    # Dew point temperature
               'TEMP',    # Temperature
               'PRES',    # Pressure
               'cbwd',    # Combined wind direction
               'Iws',     # Integrated wind speed
               'Is',      # Integrated solar radiation
               'Ir',      # Integrated infrared radiation
               'datetime']

In [ ]:
tmp_df = data_df[rel_columns].copy()
# Categorical variable 'cbwd' (wind direction)
# Convert to one-hot encoding
cbwd_dummies = pd.get_dummies(tmp_df['cbwd'], prefix='wind_dir')

tmp_df = pd.concat([tmp_df, cbwd_dummies], axis=1)

tmp_df.drop('cbwd', axis=1, inplace=True)

tmp_df = tmp_df.sort_values('datetime', axis=0, ascending=True)
tmp_df = tmp_df.reset_index(drop=True)
tmp_df.head()

In [ ]:
tmp_df.shape

In [ ]:
target_col = 'pm2.5'
rel_cols = [col for col in tmp_df.columns if col != target_col and col != 'datetime']
rel_cols

In [ ]:
num_features = len(rel_cols)
num_features

## Plotting samples

In [ ]:
fig = plt.figure(figsize = (15,8))
plot_cols = ['DEWP', 'TEMP', 'PRES',  'Iws',  'Is',  'Ir']
for i, col in enumerate(plot_cols):
    ax = fig.add_subplot(2, 3, i + 1)
    tmp_df.plot(x='datetime', y=col, style=".", ax = ax);
ax.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(tmp_df[rel_cols + [target_col]].corr().abs(), 
            annot=True,           # Show correlation values
            fmt='.2f',            # 2 decimal places
            cmap='Blues',         # Color scheme
            center=0,             # Center colormap at 0
            square=True,          # Square cells
            linewidths=0.5,       # Grid lines
            cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)
plt.title('Correlation Matrix of PRSA Air Pollution Features', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Removing Ir, Is, and Pres columns

In [ ]:
rel_cols

In [ ]:
time_steps = 24
feature_cols = ['DEWP', 'TEMP',  'Iws', 'wind_dir_NE', 'wind_dir_NW', 'wind_dir_SE', 'wind_dir_cv']

In [ ]:
X, y = [], []
for i in range(0, len(tmp_df) - time_steps, time_steps):
            # Input: time_steps of all features
        X.append(tmp_df.iloc[i:i+time_steps][feature_cols].to_numpy())
        # Target: next hour's PM2.5 value
        y.append(tmp_df.iloc[i+time_steps][target_col])
X_data = np.array(X) # # (samples, time_steps, features)
X_data = X_data[:, :-1, :] # remove last time step to avoid data leakage
y_data = np.array(y)

In [ ]:
X_data.shape, y_data.shape

In [ ]:
X_data.shape[0]*0.8/BATCH_SIZE

In [ ]:
# Calculate number of training samples
n_train_samples = 45 * BATCH_SIZE  # 45 * 32 = 1440 samples

# Method 1: Simple split based on number of samples
X_train = X_data[:n_train_samples]
y_train = y_data[:n_train_samples]
X_test = X_data[n_train_samples:]
y_test = y_data[n_train_samples:]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

#### Little work around for scaling

In [ ]:
# Scale features
n_features = X_train.shape[2]

# Reshape for scaling
X_train_2d = X_train.reshape(-1, n_features)
X_test_2d = X_test.reshape(-1, n_features)


In [ ]:
scaler = StandardScaler()
X_train_scaled_2d = scaler.fit_transform(X_train_2d)
X_test_scaled_2d = scaler.transform(X_test_2d)
X_train_scaled_2d.shape, X_test_scaled_2d.shape

In [ ]:
# Reshape back to 3D
X_train = X_train_scaled_2d.reshape(X_train.shape)
X_test = X_test_scaled_2d.reshape(X_test.shape)
X_train.shape, X_test.shape

In [ ]:
# Scale target
'''StandardScaler expects 2D input shape (n_samples, n_features), but our y_train is 1D shape (n_samples,).'''
target_scaler = StandardScaler()
y_train = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

y_train.shape, y_test.shape

## Create y_df for plotting

In [ ]:
y_idx = np.arange(time_steps, tmp_df.shape[0], time_steps)
y_df = tmp_df.iloc[y_idx][['datetime', target_col]]
y_df.head()

In [ ]:
y_df.tail()

**Note:** in regression task, gains of scaling y and not proportional

### Converting to Datasets

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test,y_test))

## Optimize for performance

train_ds = train_ds.cache()  # Cache first
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds = test_ds.cache()
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Shuffle and batch the dataset
train_ds = train_ds.shuffle(buffer_size=X_train.shape[0]).batch(BATCH_SIZE)
test_ds = test_ds.shuffle(buffer_size=X_test.shape[0]).batch(BATCH_SIZE)

In [ ]:
# Building the enhanced model with multiple RNN layers and regularization

h_units = 256  # number of units in the RNN layers

input_shape = (time_steps-1, X_train.shape[2])  # we are using two features

kernel_initializer = tf.keras.initializers.GlorotUniform(seed=RANDOM_STATE)

model = tf.keras.models.Sequential()

# Input layer
model.add(tf.keras.layers.Input(shape=input_shape))

# First SimpleRNN layer with return_sequences=True to pass sequences to the next layer
# Added recurrent_dropout for regularization to prevent overfitting
model.add(tf.keras.layers.SimpleRNN(units=h_units,
                                    kernel_initializer=kernel_initializer,
                                    activation='tanh',
                                    return_sequences=True,  # Important: returns sequences for next RNN layer
                                    dropout=0.2,           # Dropout for input regularization
                                    recurrent_dropout=0.2)) # Dropout for recurrent connections

# Second SimpleRNN layer - processes the sequences from first layer
# This second layer can capture more complex temporal dependencies
model.add(tf.keras.layers.SimpleRNN(units=h_units // 2,  # Reducing units to avoid overfitting
                                    kernel_initializer=kernel_initializer,
                                    activation='tanh',
                                    return_sequences=False,  # Only returns final output
                                    dropout=0.2,
                                    recurrent_dropout=0.2))

# Dropout layer for additional regularization
model.add(tf.keras.layers.Dropout(0.3))

# Output layer
model.add(tf.keras.layers.Dense(1,
                                kernel_initializer=kernel_initializer,
                                activation='linear'))

model.compile(loss='mean_squared_error',
              optimizer='adam',
              metrics=[tf.keras.metrics.RootMeanSquaredError()])

# Display model summary
model.summary()

In [ ]:
history = model.fit(train_ds,
                    epochs=EPOCHS, 
                    validation_data=test_ds,
                    batch_size= BATCH_SIZE, 
                    verbose=1)

In [ ]:
hist_df = pd.DataFrame(history.history)
hist_df = hist_df.rename({'root_mean_squared_error': 'rmse', 'val_root_mean_squared_error' : 'val_rmse'}, axis=1)


fn_plot_tf_hist(hist_df)
#.8172

In [ ]:
# make predictions
y_train_pred_scaled = model.predict(X_train)
y_test_pred_scaled = model.predict(X_test)
y_pred_train = target_scaler.inverse_transform(y_train_pred_scaled.reshape(-1, 1))
y_pred_test = target_scaler.inverse_transform(y_test_pred_scaled.reshape(-1, 1))
y_pred = np.append(y_pred_train, y_pred_test)

In [ ]:
res_df = y_df.copy()
res_df['pred'] = y_pred
res_df['datetime'] = res_df['datetime'].dt.date
res_df.head()

In [ ]:
res_df.tail()

In [ ]:
fig, ax = plt.subplots(figsize = (15,6))

res_df.plot(x='datetime', y=['pm2.5','pred'], ax = ax);

ax.vlines(res_df.iloc[X_train.shape[0]]['datetime'], 
          res_df['pm2.5'].min(), 
          res_df['pm2.5'].max(), color = 'k', 
          linewidth=3.0, zorder=10, alpha =0.6)

ax.grid()

### For further experiments add more relevant features:

- Lag features: PM2.5 values from previous days

- Rolling statistics: 6-hour, 12-hour, 24-hour moving averages